#Notebook to Train the Model

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts

!pip install edit_distance

Mounted at /content/drive
/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts


# Adding W&B

In [2]:
!pip install wandb

import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tkorol1 (tkorol1-ucla) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## augmentations.py
Equivalent but in Colab (same for all other cells)

In [3]:
import math
import numbers
import torch
from torch import nn
from torch.nn import functional as F

wandb.init(
    project="BCI Final Project",
    name="Revised Analysis W Text fin",
)

class WhiteNoise(nn.Module):
    def __init__(self, std=0.1):
        super().__init__()
        self.std = std

    def forward(self, x):
        noise = torch.randn_like(x) * self.std
        return x + noise

class MeanDriftNoise(nn.Module):
    def __init__(self, std=0.1):
        super().__init__()
        self.std = std

    def forward(self, x):
        _, C = x.shape
        noise = torch.randn(1, C) * self.std
        return x + noise

class GaussianSmoothing(nn.Module):
    """
    Apply gaussian smoothing on a
    1d, 2d or 3d tensor. Filtering is performed seperately for each channel
    in the input using a depthwise convolution.
    """

    def __init__(self, channels, kernel_size, sigma, dim=2):
        super(GaussianSmoothing, self).__init__()
        if isinstance(kernel_size, numbers.Number):
            kernel_size = [kernel_size] * dim
        if isinstance(sigma, numbers.Number):
            sigma = [sigma] * dim

        # The gaussian kernel is the product of the
        # gaussian function of each dimension.
        kernel = 1
        meshgrids = torch.meshgrid(
            [torch.arange(size, dtype=torch.float32) for size in kernel_size]
        )
        for size, std, mgrid in zip(kernel_size, sigma, meshgrids):
            mean = (size - 1) / 2
            kernel *= (
                1
                / (std * math.sqrt(2 * math.pi))
                * torch.exp(-(((mgrid - mean) / std) ** 2) / 2)
            )

        # Make sure sum of values in gaussian kernel equals 1.
        kernel = kernel / torch.sum(kernel)

        # Reshape to depthwise convolutional weight
        kernel = kernel.view(1, 1, *kernel.size())
        kernel = kernel.repeat(channels, *[1] * (kernel.dim() - 1))

        self.register_buffer("weight", kernel)
        self.groups = channels

        if dim == 1:
            self.conv = F.conv1d
        elif dim == 2:
            self.conv = F.conv2d
        elif dim == 3:
            self.conv = F.conv3d
        else:
            raise RuntimeError(
                "Only 1, 2 and 3 dimensions are supported. Received {}.".format(dim)
            )

    def forward(self, input):
        return self.conv(input, weight=self.weight, groups=self.groups, padding="same")

## dataset.py

In [4]:
import torch
from torch.utils.data import Dataset

class SpeechDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
        self.n_days = len(data)
        self.n_trials = sum([len(d["sentenceDat"]) for d in data])

        self.neural_feats = []
        self.phone_seqs = []
        self.neural_time_bins = []
        self.phone_seq_lens = []
        self.days = []
        for day in range(self.n_days):
            for trial in range(len(data[day]["sentenceDat"])):
                self.neural_feats.append(data[day]["sentenceDat"][trial])
                self.phone_seqs.append(data[day]["phonemes"][trial])
                self.neural_time_bins.append(data[day]["sentenceDat"][trial].shape[0])
                self.phone_seq_lens.append(data[day]["phoneLens"][trial])
                self.days.append(day)

    def __len__(self):
        return self.n_trials

    def __getitem__(self, idx):
        neural_feats = torch.tensor(self.neural_feats[idx], dtype=torch.float32)

        if self.transform:
            neural_feats = self.transform(neural_feats)

        return (
            neural_feats,
            torch.tensor(self.phone_seqs[idx], dtype=torch.int32),
            torch.tensor(self.neural_time_bins[idx], dtype=torch.int32),
            torch.tensor(self.phone_seq_lens[idx], dtype=torch.int32),
            torch.tensor(self.days[idx], dtype=torch.int64),
        )

## model.py

In [7]:
import torch
from torch import nn

# torch.nn.Module (neural network)
class GRUDecoder(nn.Module):
  #def __init__ to initiate classes
    def __init__(
        self,
        neural_dim,
        n_classes,
        hidden_dim,
        layer_dim,
        nDays=24,
        dropout=0,
        device="cuda",
        strideLen=4,
        kernelLen=14,
        gaussianSmoothWidth=0,
        bidirectional=False,
        max_mask_length=25,
        num_masks=3
    ):
        # super(Class name , "self") to initialize
        super(GRUDecoder, self).__init__()

        # Defining the number of layers and the nodes in each layer
        # self. to store parameters
        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim
        self.neural_dim = neural_dim
        self.n_classes = n_classes
        self.nDays = nDays
        self.device = device
        self.dropout = dropout
        self.strideLen = strideLen
        self.kernelLen = kernelLen
        self.gaussianSmoothWidth = gaussianSmoothWidth
        self.bidirectional = bidirectional
        self.max_mask_length = max_mask_length
        self.num_masks = num_masks
        # Softsign(x) = x / (1+|x|)
        self.inputLayerNonlinearity = torch.nn.Softsign()
        # (self.kernellen,1) - (height,width) is the kernel_size
        # torch.nn.Unfold extracts sliding local blocks from a batched input tensor
        # eg) 1-10, 2-11, 3-12, and so forth
        self.unfolder = torch.nn.Unfold(
            (self.kernelLen, 1), dilation=1, padding=0, stride=self.strideLen
        )
        self.gaussianSmoother = GaussianSmoothing(
            neural_dim, 20, self.gaussianSmoothWidth, dim=1
        )
        # torch.nn.Parameter ensures dayWeights,dayBias are trainable parameters
        self.dayWeights = torch.nn.Parameter(torch.randn(nDays, neural_dim, neural_dim))
        self.dayBias = torch.nn.Parameter(torch.zeros(nDays, 1, neural_dim))

        # torch.eye(neural_dim) returns a 2D identity matrix with dimension neural_dim
        # .data ables us to modify tensor values without affecting gradients
        for x in range(nDays):
            self.dayWeights.data[x, :, :] = torch.eye(neural_dim)

        # GRU (Gated recurrent unit) layers - refer to Lecture 12
        self.gru_decoder = nn.GRU(
            (neural_dim) * self.kernelLen,
            hidden_dim,
            layer_dim,
            batch_first=True,
            dropout=self.dropout,
            bidirectional=self.bidirectional,
        )

        """
        ###########################################################
        # Create each GRU layer separately so that we have access to intermediate layers
        self.gru_layers = nn.ModuleList()
        for i in range(layer_dim):
          input_dim = neural_dim * self.kernelLen if i == 0 else hidden_dim
          self.gru_layers.append(nn.GRU(
              input_dim,
              hidden_dim,
              batch_first=True,
              dropout=self.dropout,
              bidirectional=self.bidirectional,
          )
          )
        ############################################################
        """

        # named_parameters() returns all trainable parameters in the GRU and their corresponding names
        for name, param in self.gru_decoder.named_parameters():
            # Learnable hidden to hidden weights
            # Orthogonal -> Identity matrix
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            # Learnable input to hidden weights
            # Xavier uniform distribution U(-a,a) where a = gain x sqrt(6/(n_inputs+n_outputs))
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)

        # Input layers
        # Output = self.inpLayer0 = nn.Linear(neural_dim,neural_dim)
        #          self.inpLayer1 = nn.Linear(neural_dim,neural_dim)
        #          and so on
        # torch.nn.Linear applies linear transformation to the data
        for x in range(nDays):
            setattr(self, "inpLayer" + str(x), nn.Linear(neural_dim, neural_dim))

        for x in range(nDays):
            thisLayer = getattr(self, "inpLayer" + str(x))
            thisLayer.weight = torch.nn.Parameter(
                thisLayer.weight + torch.eye(neural_dim)
            )
        ###################################################
        # Layer normalization
        if self.bidirectional:
            self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        else:
            self.layer_norm = nn.LayerNorm(hidden_dim)
        ###################################################

        # rnn outputs
        if self.bidirectional:
            self.fc_decoder_out = nn.Linear(
                hidden_dim * 2, n_classes + 1
            )  # +1 for CTC blank
        else:
            self.fc_decoder_out = nn.Linear(hidden_dim, n_classes + 1)  # +1 for CTC blank


    def forward(self, neuralInput, dayIdx):
        # torch.permute -> permutes the dimensions of the tensor... (a,b,c) to (0,2,1) would be (a,c,b)
        neuralInput = torch.permute(neuralInput, (0, 2, 1))
        neuralInput = self.gaussianSmoother(neuralInput)
        neuralInput = torch.permute(neuralInput, (0, 2, 1))

        ######################################################
        # Apply time mask
        if self.training:
          neuralInput = self.apply_time_masking(neuralInput)
        ######################################################

        # apply day layer
        # 0 refers to 1st dimension (rows) for torch.index_select
        dayWeights = torch.index_select(self.dayWeights, 0, dayIdx)
        # Batch matrix multiplication... neuralInput with shape b,t,d and dayWeights with shape b,d,k results
        # in a tensor with shape b,t,k
        transformedNeural = torch.einsum(
            "btd,bdk->btk", neuralInput, dayWeights
        ) + torch.index_select(self.dayBias, 0, dayIdx)
        # Apply nonlinearity to ensure complexity to the model
        transformedNeural = self.inputLayerNonlinearity(transformedNeural)

        # stride/kernel
        # torch.unsqueeze( , 3) ensures 2D structure needed for torch.nn.Unfold
        stridedInputs = torch.permute(
            self.unfolder(
                torch.unsqueeze(torch.permute(transformedNeural, (0, 2, 1)), 3)
            ),
            (0, 2, 1),
        )

        # apply RNN layer
        # device = self.device moves tensor to whatever device being used
        # .requires_grad_() ensures gradients are computed for the tensor
        if self.bidirectional:
            h0 = torch.zeros(
                self.layer_dim * 2,
                transformedNeural.size(0),
                self.hidden_dim,
                device=self.device,
            ).requires_grad_()
        else:
            h0 = torch.zeros(
                self.layer_dim,
                transformedNeural.size(0),
                self.hidden_dim,
                device=self.device,
            ).requires_grad_()

        # self.gru_decoder = nn.GRU() defined earlier
        # h0.detach() detaches h0 from the computational graph (generates new tensor that does not require gradient)
        hid, _ = self.gru_decoder(stridedInputs, h0.detach())

        ################################################
        # Apply layer normalization before output
        hid_norm = self.layer_norm(hid)
        ################################################

        # get seq
        # self.fc_decoder_out = nn.Linear() defined earlier - RNN output
        seq_out = self.fc_decoder_out(hid_norm)
        return seq_out

        """
        #################################################
        # Separate the first layer from the rest (different shape)
        hid, _ = self.gru_layers[0](stridedInputs, h0[0:1].detach())

        for i in range(1, self.layer_dim):
          hid, _ = self.gru_layers[i](hid, h0[i:i+1].detach())
          # We chose the 3rd layer to be the intermediate layer
          if i==2:
            hid_intermediate = hid.clone()
            hid_norm_int = self.layer_norm(hid_intermediate)
            seq_out_inter = self.fc_decoder_out(hid_norm_int)
        hid_norm = self.layer_norm(hid)
        seq_out_main = self.fc_decoder_out(hid_norm)
        #################################################
        return seq_out_main, seq_out_inter
        """

    #########################################################################
    # Define time masking function
    def apply_time_masking(self,input_tensor):
      batch_size, time_steps, channels = input_tensor.shape
      for batch in range(batch_size):
        for _ in range(self.num_masks):
          mask_length = torch.randint(1, self.max_mask_length+1, (1,)).item()
          max_start = max(1, time_steps - mask_length)
          start_time = torch.randint(0, max_start, (1,)).item()
          input_tensor[batch, start_time:start_time+mask_length, :] = 0
      return input_tensor
    ##########################################################################

## neural_decoder_trainer.py

In [8]:
# Code to convert phoneme indices to actual phonemes

# Decode into Phonemes for inspection:
def phoneme_conv(phoneme_list):
  PHONE_DEF = [
      'AA', 'AE', 'AH', 'AO', 'AW',
      'AY', 'B',  'CH', 'D', 'DH',
      'EH', 'ER', 'EY', 'F', 'G',
      'HH', 'IH', 'IY', 'JH', 'K',
      'L', 'M', 'N', 'NG', 'OW',
      'OY', 'P', 'R', 'S', 'SH',
      'T', 'TH', 'UH', 'UW', 'V',
      'W', 'Y', 'Z', 'ZH'
  ]
  PHONE_DEF_SIL = PHONE_DEF + ['SIL']

  # Go through phoneme indices and display the corresponding phonemes:

  phoneme_text = ""

  # If phoneme list is empty (early runs have bad decoding) we return empty
  if not phoneme_list:
    return("")

  else:

    for index in phoneme_list:

      # Need to use -1 since the phonemes are indexed by 1 (and 0 is end token)

      if index == 0:
        break

      phoneme = PHONE_DEF_SIL[index-1]

      phoneme_text += f'{phoneme} '

    return(phoneme_text.strip())

In [9]:
import torch
import numpy as np
import pickle
import random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

MODEL_REPO_ID = "tonykorol/t5_phoneme_decoder"
DATA_PATH = '/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/processed_data/ptDecoder_ctc.pkl'

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the model from my huggingface repo
hf_tokenizer = AutoTokenizer.from_pretrained("t5-small")
hf_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO_ID)
hf_model.to(device)

# Adding the full decoding pipeline as tested in the T5 training code:

def decode_from_indices(
    phoneme_indices: list,
    hf_tokenizer: AutoTokenizer,
    hf_model: AutoModelForSeq2SeqLM,
    device: str) -> str:

    # Convert from indices to phoneme string

    phoneme_sequence = phoneme_conv(phoneme_indices)

    # Add the same prompt to the beginning of phonemes for T5 decoding

    input_text = "Transcribe phonemes to standard English: " + phoneme_sequence

    # Tokenize the conditioned input string

    current_input_ids = hf_tokenizer(
        input_text,
        return_tensors="pt",
        padding='max_length',
        max_length=256
    ).input_ids.to(device)

    # Generate the output sequence

    predicted_ids = hf_model.generate(

        # Had to mess with these params quite a lot to get an accurate output

        current_input_ids,
        max_length=128,
        early_stopping=True,
        #repetition_penalty=1.5,
        num_beams=5,
        do_sample=False,
        #length_penalty=0.2,
    )

    # Decode the predicted tokens to string format

    predicted_text = hf_tokenizer.decode(
        predicted_ids.squeeze(),
        skip_special_tokens=True
    )

    return predicted_text

# Implement the complete decoding block which combines the above steps

def full_decoding_pipeline(phoneme_inds):

    # Convert the indices to phonemes that can be used by the model

    clean_phoneme_seq = phoneme_conv(phoneme_inds)

    # Display the actual input for the T5 decoding

    input_text_for_display = f"Transcribe phonemes to standard English: {clean_phoneme_seq}"

    # call the decoding function to get the output and display

    decoded_text = decode_from_indices(
        phoneme_indices=phoneme_inds,
        hf_tokenizer=hf_tokenizer,
        hf_model=hf_model,
        device=device
    )

    print(f" \nDecoding Results")
    print(f"Fine Tuned LLM Used:             {MODEL_REPO_ID}")
    print(f"Input Phonemes:         {input_text_for_display}")
    print(f"Decoded Text:     {decoded_text}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

In [19]:
import os
import pickle
import time

from edit_distance import SequenceMatcher
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR, ConstantLR


def getDatasetLoaders(
    datasetName,
    batchSize,
):
    with open(datasetName, "rb") as handle:
        loadedData = pickle.load(handle)

    def _padding(batch):
        X, y, X_lens, y_lens, days = zip(*batch)
        X_padded = pad_sequence(X, batch_first=True, padding_value=0)
        y_padded = pad_sequence(y, batch_first=True, padding_value=0)

        return (
            X_padded,
            y_padded,
            torch.stack(X_lens),
            torch.stack(y_lens),
            torch.stack(days),
        )

    train_ds = SpeechDataset(loadedData["train"], transform=None)
    test_ds = SpeechDataset(loadedData["test"])

    train_loader = DataLoader(
        train_ds,
        batch_size=batchSize,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=_padding,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batchSize,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=_padding,
    )

    return train_loader, test_loader, loadedData

#############################################################################
# Implementing CTC Loss with label smoothing
def ctc_with_label_smoothing(input, target, input_length, target_length, eps=0.1):
    """
    Args:
        input: Prediction tensor from model.forward(X, dayIdx)
        target: target sequences with shape (N,S) where S=target sequence length
        input_length: length of input with shape (N,)
        target_length: length of target with shape (N,)
        eps: smoothing coefficient
    """
    log_prob = torch.permute(input.log_softmax(2), [1,0,2])

    loss_ctc = torch.nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)
    std_loss = loss_ctc(log_prob, target, input_length, target_length)

    prob = torch.exp(log_prob)
    entropy = - (prob * log_prob).sum(dim = -1).mean()

    new_loss = ((1 - eps) * std_loss) + (eps * entropy)
    return new_loss

#############################################################################

def trainModel(args):
    os.makedirs(args["outputDir"], exist_ok=True)
    torch.manual_seed(args["seed"])
    np.random.seed(args["seed"])
    device = "cuda"

    with open(args["outputDir"] + "/args", "wb") as file:
        pickle.dump(args, file)

    trainLoader, testLoader, loadedData = getDatasetLoaders(
        args["datasetPath"],
        args["batchSize"],
    )

    model = GRUDecoder(
        neural_dim=args["nInputFeatures"],
        n_classes=args["nClasses"],
        hidden_dim=args["nUnits"],
        layer_dim=args["nLayers"],
        nDays=len(loadedData["train"]),
        dropout=args["dropout"],
        device=device,
        strideLen=args["strideLen"],
        kernelLen=args["kernelLen"],
        gaussianSmoothWidth=args["gaussianSmoothWidth"],
        bidirectional=args["bidirectional"],
    ).to(device)

    loss_ctc = torch.nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)
    # Experiment with AdamW optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args["lrStart"],
        betas=(0.9, 0.999),
        eps=0.05,
        weight_decay=args["l2_decay"],
    )

    ###################################################
    # Modified learning rate
    # Warm-up -> Hold -> Cosine decay
    num_steps = args["nBatch"]
    warmup_steps = num_steps / 20 # Results in step of 500
    decay_step = (0.7/1.0) * num_steps

    warmup = LinearLR(
            optimizer,
            start_factor = 1 / warmup_steps,
            total_iters = warmup_steps,
            )
    constant = ConstantLR(
            optimizer,
            factor = 1.0,
            total_iters = decay_step - warmup_steps,
            )
    cosine = CosineAnnealingLR(
            optimizer,
            T_max = num_steps - decay_step,
            eta_min = args["lrStart"] / 10
            )
    scheduler = SequentialLR(
            optimizer,
            schedulers = [warmup, constant, cosine],
            milestones = [warmup_steps, decay_step],
            )
    ###################################################

    # Adding lists to store the decoded output per 1000 batches for visualization

    test_dec_seq = []
    test_true_seq = []

    # --train--
    testLoss = []
    testCER = []

    # Log training loss as well

    trainLoss = []
    trainCER = []

    max_norm = 1

    ###########################################
    # Initializing variables for early stopping
    best_cer = float("inf") # Set to arbitrarily large number such that first cer is the best_cer
    patience = 10 # Number of batches (in hundreds) to compare to before stopping
    patience_counter = 0
    ###########################################

    for batch in range(args["nBatch"]):
        model.train()

        X, y, X_len, y_len, dayIdx = next(iter(trainLoader))
        X, y, X_len, y_len, dayIdx = (
            X.to(device),
            y.to(device),
            X_len.to(device),
            y_len.to(device),
            dayIdx.to(device),
        )

        # Noise augmentation is faster on GPU
        if args["whiteNoiseSD"] > 0:
            X += torch.randn(X.shape, device=device) * args["whiteNoiseSD"]

        if args["constantOffsetSD"] > 0:
            X += (
                torch.randn([X.shape[0], 1, X.shape[2]], device=device)
                * args["constantOffsetSD"]
            )

        ####################################################################
        # Use the function for CTC Loss with label smoothing
        pred = model.forward(X, dayIdx)
        x_len_mod = ((X_len - model.kernelLen) / model.strideLen).to(torch.int32)

        loss = ctc_with_label_smoothing(pred, y, x_len_mod, y_len)
        loss = torch.sum(loss)
        #####################################################################

        """
        #########################################################
        # Implementation of intermediate CTC Loss
        pred_main,pred_inter = model.forward(X, dayIdx)

        loss_main = loss_ctc(
            torch.permute(pred_main.log_softmax(2), [1, 0, 2]),
            y,
            ((X_len - model.kernelLen) / model.strideLen).to(torch.int32),
            y_len,
        )
        loss_main = torch.sum(loss_main)

        # Include CTC loss for intermediate layer
        loss_inter = loss_ctc(
            torch.permute(pred_inter.log_softmax(2), [1, 0, 2]),
            y,
            ((X_len - model.kernelLen) / model.strideLen).to(torch.int32),
            y_len
        )
        loss_inter = torch.sum(loss_inter)

        # Use weights to control the contribution of each loss
        w = 0.3
        loss = (1-w) * loss_main + w * loss_inter
        #########################################################
        """

        #####################################################################
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        # Apply gradient clipping
        #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        optimizer.step()
        scheduler.step()
        #####################################################################

        trainLoss.append(loss.item())

        # Calculate train CER the same way as val cer is done below

        with torch.no_grad():
            decoded_total_edit = 0
            decoded_total_len = 0
            adjustedLens = ((X_len - model.kernelLen) / model.strideLen).to(torch.int32)
            for iterIdx in range(pred.shape[0]):
                decodedSeq = torch.argmax(pred[iterIdx, 0:adjustedLens[iterIdx], :], dim=-1)
                decodedSeq = torch.unique_consecutive(decodedSeq, dim=-1)
                decodedSeq = decodedSeq.cpu().detach().numpy()
                decodedSeq = np.array([i for i in decodedSeq if i != 0])
                trueSeq = np.array(y[iterIdx][0:y_len[iterIdx]].cpu().detach())
                matcher = SequenceMatcher(a=trueSeq.tolist(), b=decodedSeq.tolist())
                decoded_total_edit += matcher.distance()
                decoded_total_len += len(trueSeq)
            traincer = decoded_total_edit / decoded_total_len
            trainCER.append(traincer)

        # Eval
        if batch == 0 or batch % 10 == 0:
            with torch.no_grad():
                model.eval()
                allLoss = []
                total_edit_distance = 0
                total_seq_length = 0
                it = 0
                for X, y, X_len, y_len, testDayIdx in testLoader:
                    X, y, X_len, y_len, testDayIdx = (
                        X.to(device),
                        y.to(device),
                        X_len.to(device),
                        y_len.to(device),
                        testDayIdx.to(device),
                    )

                    pred = model.forward(X, testDayIdx)
                    loss = loss_ctc(
                        torch.permute(pred.log_softmax(2), [1, 0, 2]),
                        y,
                        ((X_len - model.kernelLen) / model.strideLen).to(torch.int32),
                        y_len,
                    )
                    loss = torch.sum(loss)

                    allLoss.append(loss.cpu().detach().numpy())

                    adjustedLens = ((X_len - model.kernelLen) / model.strideLen).to(
                        torch.int32
                    )
                    for iterIdx in range(pred.shape[0]):
                        decodedSeq = torch.argmax(
                            torch.tensor(pred[iterIdx, 0 : adjustedLens[iterIdx], :]),
                            dim=-1,
                        )  # [num_seq,]
                        decodedSeq = torch.unique_consecutive(decodedSeq, dim=-1)
                        decodedSeq = decodedSeq.cpu().detach().numpy()
                        decodedSeq = np.array([i for i in decodedSeq if i != 0])

                        trueSeq = np.array(
                            y[iterIdx][0 : y_len[iterIdx]].cpu().detach()
                        )

                        matcher = SequenceMatcher(
                            a=trueSeq.tolist(), b=decodedSeq.tolist()
                        )
                        total_edit_distance += matcher.distance()
                        total_seq_length += len(trueSeq)

                    if batch==1 or batch%1000 ==0:
                      # Print a few natural language results every 1000 batches

                      # We should see evolution of the GRU output to slowly match the ground truth

                      if (it < 5):
                        a = trueSeq.tolist()
                        b = decodedSeq.tolist()

                        test_true_seq.append(a)
                        test_dec_seq.append(b)

                        print(f'\n Batch {batch} true: \n')
                        print(a)
                        print(f'\n Batch {batch} decoded: \n')
                        print(b)
                        print(f'\n ------------ Batch {batch} Ground Truth Phoneme to Text with Fine-Tuned LLM ------------- \n')
                        full_decoding_pipeline(a)
                        print(f'\n ------------ Batch {batch} Modified GRU Decoded Phoneme to Text with Fine-Tuned LLM ------------  \n')
                        full_decoding_pipeline(b)
                      it +=1

                val_loss = np.sum(allLoss) / len(testLoader)
                val_cer = total_edit_distance / total_seq_length

                testCER.append(val_cer)
                testLoss.append(val_loss)

                avgDayLoss = np.sum(allLoss) / len(testLoader)
                cer = total_edit_distance / total_seq_length

                print(
                    f"batch {batch}, ctc loss: {avgDayLoss:>7f}, cer: {cer:>7f}"
                )
                #####################################
                # Apply early stopping
                if cer < best_cer:
                    best_cer = cer
                    patience_counter = 0
                else:
                    patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping *would be* activated")
                    patience_counter = 0
                ######################################
                print(f"Patience counter: {patience_counter}")
                startTime = time.time()
            """
            if len(testCER) > 0 and cer < np.min(testCER):
                torch.save(model.state_dict(), args["outputDir"] + "/modelWeights")
            """

            wandb.log({
                "batch": batch,
                "train_loss": trainLoss[-1],
                "train_CER": trainCER[-1],
                "val_loss": val_loss,
                "val_CER": val_cer,
                "lr": scheduler.get_last_lr()[0]
            })

            testLoss.append(avgDayLoss)
            testCER.append(cer)

            tStats = {}
            tStats["testLoss"] = np.array(testLoss)
            tStats["testCER"] = np.array(testCER)

            with open(args["outputDir"] + "/trainingStats", "wb") as file:
                pickle.dump(tStats, file)

## train_model.py

In [20]:

modelName = 'speechBaseline4'

args = {}
args['outputDir'] = '/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/Training_Results' + modelName
args['datasetPath'] = '/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/processed_data/ptDecoder_ctc.pkl'
args['seqLen'] = 150
args['maxTimeSeriesLen'] = 1200
args['batchSize'] = 64
args['lrStart'] = 0.04
args['lrEnd'] = 0.02
args['nUnits'] = 1024
args['nBatch'] = 12000 #3000
args['nLayers'] = 5
args['seed'] = 0
args['nClasses'] = 40
args['nInputFeatures'] = 256
args['dropout'] = 0.4
args['whiteNoiseSD'] = 0.8
args['constantOffsetSD'] = 0.2
args['gaussianSmoothWidth'] = 2.0
args['strideLen'] = 4
args['kernelLen'] = 32
args['bidirectional'] = False
args['l2_decay'] = 1e-5

trainModel(args)

/tmp/ipython-input-617550283.py:293: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(pred[iterIdx, 0 : adjustedLens[iterIdx], :]),



 Batch 0 true: 

[17, 23, 31, 11, 21, 3, 19, 3, 23, 29, 40, 19, 2, 7, 9, 40, 2, 31, 40, 16, 17, 22, 40, 3, 20, 37, 34, 38, 17, 24, 21, 18, 40]

 Batch 0 decoded: 

[31, 33, 9, 12, 7, 40, 27, 31]

 ------------ Batch 0 Ground Truth Phoneme to Text with Fine-Tuned LLM ------------- 

 
Decoding Results
Fine Tuned LLM Used:             tonykorol/t5_phoneme_decoder
Input Phonemes:         Transcribe phonemes to standard English: IH N T EH L AH JH AH N S SIL JH AE B D SIL AE T SIL HH IH M SIL AH K Y UW Z IH NG L IY SIL
Decoded Text:     Individuals just at his own pace.

 ------------ Batch 0 Modified GRU Decoded Phoneme to Text with Fine-Tuned LLM ------------  

 
Decoding Results
Fine Tuned LLM Used:             tonykorol/t5_phoneme_decoder
Input Phonemes:         Transcribe phonemes to standard English: T UH D ER B SIL P T
Decoded Text:     Todder.

 Batch 0 true: 

[17, 14, 40, 10, 11, 28, 40, 12, 5, 23, 9, 40, 11, 23, 18, 22, 4, 28, 40]

 Batch 0 decoded: 

[31, 3, 31, 40, 17, 36, 12

KeyboardInterrupt: 